# Step 4: Model 2 (long-term) DIN training

Trains the long-term model described in Section 2.2 of the report: a **DIN-style network with one target-attention block per history channel** (click ≤ 7 days, cart ≤ 90 days, order ≤ 365 days).

| | |
|---|---|
| **Inputs** | `seq_<split>.parquet` (Sequence sequence inputs) and `features_<split>.parquet` (Scalar scalar features) |
| **Objective** | same as Model 1: binary on "any interaction" with sample weights click 1, cart 5, purchase 10 (1 for non-interacted candidates) |
| **Splits** | `train` (negative-sampled) fits the model; `val1` early stopping and tuning; `val2` calibration (and the gate, in its own notebook); `test` once, at the end |
| **Model selection** | early stopping (val1 checked twice per epoch) and the overfitting tuning in Step 6, on val1 NDCG@10 (purchase relevance, per search) |
| **Evaluation** | per-search NDCG@10, HitRate@10, Hit@1, MRR on searches with a purchase; metric code identical to Model 1 and the LightGBM ablation |
| **Outputs** | model checkpoint, raw and calibrated P2 for every val2 and test candidate (for the gate), per-search metrics, diagnostics, optional tuning and input ablation |

Test is scored once, in Step 8. Settings are chosen on val1 and calibration is fitted on val2, so neither of them sees test, and calibration does not reuse the searches used for model selection.

**Runtime:** Colab with a GPU (Runtime → Change runtime type → T4 GPU). Step 4 runs quick checks of the model on a few searches before the full training starts.

**Run order:** Step 1 (`step1_data_preprocessing`), then the Model 2 sequence inputs (`step2_model2_feature_engineering`) and scalar features (Dom), then this notebook.

## Setup

In [6]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set the paths in Settings to local folders.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:

# !pip install optuna
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from sklearn.linear_model import LogisticRegression

import time
import optuna
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU: training will be very slow")
print("numpy", np.__version__, "| pandas", pd.__version__, "| pyarrow", pa.__version__)

torch 2.14.1+cu130 | device: cuda | Tesla T4
numpy 2.4.6 | pandas 2.2.3 | pyarrow 23.0.1


## Settings

`CFG` holds every modelling choice. The optional tuning in Step 6 replaces a few of them with the best values found on val1. The ablation in Step 10 reruns training with one of the `use_*` switches (or one channel) turned off, so the reference model and the ablations differ in exactly one input group.

In [8]:
# >>> Paths: change to where the Sequence and Scalar outputs live <<<
PROJECT = PROJECT = Path("/content/drive/MyDrive/Grp8Submission/pipeline_data")
SEQ_DIR = PROJECT / "step2_model2_seq"        # sequence inputs
SCALAR_DIR = PROJECT / "step2_model2_scalar"  # scalar features
OUT_DIR = PROJECT / "step4_model2"            # this notebook's outputs

SPLITS = ["train", "val1", "val2", "test"]
TUNE_SPLIT = "val1"     # early stopping, tuning, ablation
CALIB_SPLIT = "val2"    # Platt calibration (and the gate, in its own notebook)
# Stop early if the shared folder is not visible (otherwise new empty folders would be
# created in your own My Drive): add a shortcut to the shared Grp8Submission folder in My Drive.
for d in (SEQ_DIR, SCALAR_DIR, OUT_DIR.parent):
    assert d.is_dir(), f"{d} not found: check the shortcut to the shared Grp8Submission folder"
OUT_DIR.mkdir(exist_ok=True)

CFG = dict(
    # inputs (switched off one at a time in the ablation)
    channels=["click", "cart", "order"],
    use_query_terms=True,      # words of the searches behind past actions
    use_name_terms=True,       # words of the candidate's product name
    use_attributes=True,       # brand / L4 category / shop embeddings (False = item ID only)
    use_decay=True,            # per-channel recency decay inside attention
    use_scalars=True,          # Scalar scalar features
    use_cand_item_id=True,     # candidate's own item embedding (False = brand / category / shop / name only)
    use_popularity=True,      # candidate popularity in train histories, per channel (log counts)
    # architecture
    emb_dim=64, att_hidden=64, mlp_hidden=(256, 128), dropout=0.2,   # emb_dim 64: best in the first tuning run
    emb_dropout=0.0,           # dropout on product and word embeddings
    # training
    lr=1e-3, weight_decay=0.0, grad_clip=5.0,
    emb_weight_decay=0.0,      # decoupled weight decay (AdamW) on the embedding tables only
    batch_cands=4096,          # candidates per training batch (whole searches are kept together)
    eval_batch_cands=8192,
    max_epochs=8,
    evals_per_epoch=2,         # check val1 twice per epoch: the best point comes early
    patience=3,                # stop after this many val1 checks without improvement
    amp=True,                  # mixed precision on GPU
    seed=0,
)
GRADED_WEIGHT = {0: 1.0, 1: 1.0, 2: 5.0, 3: 10.0}   # label -> sample weight (interim report, as Model 1)

SEQ_FIELDS = ["item", "brand_id", "cate_id_4", "shop_id", "action", "time_bucket", "has_query", "qry_terms", "days"]
ATTR_COLS = ["brand_id", "cate_id_4", "shop_id"]

## Step 1: Load the inputs

`load_split` is the reader from Sequence: one parquet file per split, one row per search, turned back into per-channel grids `[searches, cap]` and flat candidate arrays (`row` = the candidate's search, `pos` = its position in the step 1 list).

In [9]:
VOCAB = json.loads((SEQ_DIR / "vocab.json").read_text())
CHANNELS = {ch: (v["actions"], v["window_days"]) for ch, v in VOCAB["channels"].items()}
HALF_LIFE = {ch: max(v["half_life_days"], 1e-3) for ch, v in VOCAB["channels"].items()}


def to_grid(arr):
    """Arrow (nested) fixed-size list array -> numpy array [n, k] or [n, k, m]."""
    shape, t = [len(arr)], arr.type
    while pa.types.is_fixed_size_list(t):
        shape.append(t.list_size)
        arr, t = arr.flatten(), t.value_type
    return arr.to_numpy(zero_copy_only=False).reshape(shape)


def load_split(name, out_dir=None):
    """(slate_id, user, seqs, cand) exactly as saved by Sequence."""
    out_dir = out_dir or SEQ_DIR
    t = pq.read_table(out_dir / f"seq_{name}.parquet")
    user, seqs, cand = {}, {ch: {} for ch in CHANNELS}, {}
    for col in t.column_names:
        arr = t.column(col).combine_chunks()
        prefix, _, field = col.partition("_")
        if prefix in CHANNELS:
            seqs[prefix][field] = to_grid(arr) if pa.types.is_fixed_size_list(arr.type) \
                                  else arr.to_numpy(zero_copy_only=False)
        elif prefix == "cand":
            values = arr.flatten()
            cand[field] = to_grid(values) if pa.types.is_fixed_size_list(values.type) else values.to_numpy()
            if "row" not in cand:
                counts = arr.value_lengths().to_numpy()
                cand["row"] = np.repeat(np.arange(len(arr)), counts).astype(np.int32)
                cand["pos"] = (np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)).astype(np.int16)
        elif col != "slate_id":
            user[col] = arr.to_numpy()
    return t.column("slate_id").to_numpy(), user, seqs, cand


RAW = {split: load_split(split) for split in SPLITS}
for split, (sid, user, seqs, cand) in RAW.items():
    caps = {ch: s["item"].shape[1] for ch, s in seqs.items()}
    print(f"{split}: {len(sid):,} searches | {len(cand['item']):,} candidates | "
          f"purchases {int((cand['label'] == 3).sum()):,} | channel caps {caps}")

train: 122,083 searches | 2,608,369 candidates | purchases 14,774 | channel caps {'click': 65, 'cart': 80, 'order': 49}
val1: 17,799 searches | 1,589,673 candidates | purchases 2,016 | channel caps {'click': 65, 'cart': 80, 'order': 49}
val2: 16,924 searches | 1,511,179 candidates | purchases 1,981 | channel caps {'click': 65, 'cart': 80, 'order': 49}
test: 17,025 searches | 1,500,706 candidates | purchases 2,065 | channel caps {'click': 65, 'cart': 80, 'order': 49}


## Step 2: Scalar features

Two sources, one value per candidate:

- **Scalar features** (`scalar_<split>.parquet`, one row per search with one list per feature, candidates in step 1 order). They are matched to the sequence inputs by `slate_id`, and every search must have the same number of candidates in both files.
- **Per-channel summaries from Sequence**: each channel's action count and days since its last action, whether the channel is empty, the total history length, and the candidate's `has_metadata` flag.
- **Candidate popularity** (used only with `use_popularity=True`): how often the candidate appears in **training** histories, per channel, as in Model 1's popularity features. Unknown items get 0.

Transformations are fitted on **train only**: features outside [0, 1] are log-transformed (signed `log1p`), every feature is standardised with the train mean and standard deviation, missing values become 0 after standardisation, and a missing-value flag is added for every feature that has missing values in train.

In [10]:
def read_scalar(split, slate_id, cand_counts):
    """Load candidate scalar features, aligned to sequence slate/candidate order."""
    path = SCALAR_DIR / f"scalar_{split}.parquet"

    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found: set SCALAR_DIR, "
            "or CFG['use_scalars'] = False to train without it"
        )

    t = pq.read_table(path)
    sid = t.column("slate_id").to_numpy()
    pos = pd.Series(np.arange(len(sid)), index=sid)

    if pos.index.duplicated().any():
        raise ValueError(f"{split}: duplicate slate_id in Scalar file")

    missing = np.setdiff1d(slate_id, sid)
    if len(missing):
        raise ValueError(
            f"{split}: {len(missing):,} searches missing from Scalar file"
        )

    order = pa.array(pos.loc[slate_id].to_numpy(), type=pa.int64())
    out = {}

    def extract(name, values):
        # A candidate struct contains one field per scalar feature.
        if pa.types.is_struct(values.type):
            for field in values.type:
                child_name = f"{name}__{field.name}" if name else field.name
                # Propagates null parent structs to their child values.
                child = pc.struct_field(values, field.name)
                extract(child_name, child)
            return

        if not (
            pa.types.is_integer(values.type)
            or pa.types.is_floating(values.type)
            or pa.types.is_boolean(values.type)
            or pa.types.is_decimal(values.type)
            or pa.types.is_null(values.type)
        ):
            raise TypeError(
                f"{split}: feature '{name}' has unsupported type "
                f"{values.type}; expected a numeric scalar"
            )

        if name in out:
            raise ValueError(f"{split}: duplicate feature name '{name}'")

        out[name] = (
            pc.cast(values, pa.float64())
            .to_numpy(zero_copy_only=False)
            .astype(np.float64)
        )

    for col in t.column_names:
        if col == "slate_id":
            continue

        arr = pc.take(t.column(col).combine_chunks(), order)

        if not (
            pa.types.is_list(arr.type)
            or pa.types.is_large_list(arr.type)
            or pa.types.is_fixed_size_list(arr.type)
        ):
            raise TypeError(
                f"{split}: column '{col}' has type {arr.type}; "
                "expected one candidate list per slate"
            )

        if arr.null_count:
            raise ValueError(
                f"{split}: column '{col}' contains null candidate lists"
            )

        lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)
        if not np.array_equal(lens, cand_counts):
            raise ValueError(
                f"{split}: '{col}' has a different number of "
                "candidates than the sequence inputs"
            )

        values = pc.list_flatten(arr)

        if pa.types.is_struct(values.type):
            # Keep the original feature names for channel filtering.
            extract("", values)
        else:
            extract(col, values)

    return out


def item_popularity():
    """Times each known item appears in TRAIN histories, per channel (train-only statistic).
    Unknown (1) and padding (0) get 0. Counts come from the capped channel grids."""
    pop = {}
    for ch, s in RAW["train"][2].items():
        c = np.bincount(s["item"].ravel(), minlength=VOCAB["n_items"]).astype(np.float64)
        c[:2] = 0.0
        pop[ch] = c
    return pop


POPULARITY = item_popularity()


def sequence_summaries(user, seqs, cand):
    """Per-candidate copies of the Sequence per-channel summaries."""
    row = cand["row"]
    # raw counts and days: ScalarTransform applies log1p once (they are outside [0, 1])
    out = {"history_len": user["history_len"].astype(np.float64)[row],
           "cand_has_metadata": cand["has_metadata"].astype(np.float64)}
    for ch, s in seqs.items():
        # out[f"{ch}_count"] = s["full_len"].astype(np.float64)[row]
        out[f"{ch}_days_since_last"] = s["days_since_last"].astype(np.float64)[row]   # NaN if the channel is empty
        out[f"{ch}_empty"] = (s["full_len"] == 0).astype(np.float64)[row]
        out[f"{ch}_popularity"] = POPULARITY[ch][cand["item"]]                    # candidate popularity (raw count)
    return out


def raw_scalars(split):
    sid, user, seqs, cand = RAW[split]
    feats = {f"sequence__{k}": v for k, v in sequence_summaries(user, seqs, cand).items()}
    counts = np.bincount(cand["row"], minlength=len(sid))
    if CFG["use_scalars"] or SCALAR_DIR.exists():
        try:
            feats.update({f"scalar__{k}": v for k, v in read_scalar(split, sid, counts).items()})
        except FileNotFoundError as e:
            if CFG["use_scalars"]:
                raise
            print(e)
    return feats


class ScalarTransform:
    """Fitted on train: signed log1p for features outside [0, 1], standardise, NaN -> 0 plus a flag."""
    def fit(self, feats):
        self.names, self.log, self.mean, self.std, self.flag = [], {}, {}, {}, []
        for k, v in feats.items():
            finite = v[np.isfinite(v)]
            self.log[k] = bool(len(finite)) and (finite.min() < 0 or finite.max() > 1)
            x = self._log(k, v)
            xf = x[np.isfinite(x)]
            self.mean[k] = float(xf.mean()) if len(xf) else 0.0
            sd = float(xf.std()) if len(xf) else 0.0
            self.std[k] = sd if sd > 1e-12 else 1.0
            self.names.append(k)
            if np.isnan(v).any():
                self.flag.append(k)
        self.columns = self.names + [f"{k}__missing" for k in self.flag]
        return self

    def _log(self, k, v):
        return np.sign(v) * np.log1p(np.abs(v)) if self.log[k] else v

    def transform(self, feats):
        if set(feats) != set(self.names):
            raise ValueError(f"scalar features differ from train: {set(feats) ^ set(self.names)}")
        cols = []
        for k in self.names:
            x = (self._log(k, feats[k]) - self.mean[k]) / self.std[k]
            cols.append(np.where(np.isfinite(x), x, 0.0))
        cols += [np.isnan(feats[k]).astype(np.float64) for k in self.flag]
        return np.column_stack(cols).astype(np.float32)


raw_sc = {split: raw_scalars(split) for split in RAW}
SCALER = ScalarTransform().fit(raw_sc["train"])
SCALARS = {split: SCALER.transform(f) for split, f in raw_sc.items()}
del raw_sc
SCALAR_NAMES = SCALER.columns
n_scalar = sum(c.startswith("scalar__") and not c.endswith("__missing") for c in SCALAR_NAMES)
print(f"{len(SCALAR_NAMES)} scalar inputs: {n_scalar} from Scalar, "
      f"{sum(c.startswith('sequence__') and not c.endswith('__missing') for c in SCALAR_NAMES)} Sequence summaries, "
      f"{len(SCALER.flag)} missing-value flags:")
for i, name in enumerate(SCALAR_NAMES, start=1):
    print(f"{i:2d}. {name}")


def scalar_columns(cfg):
    """Column indices of the scalar inputs used under a config (removed channels drop their scalars too)."""
    dropped = [ch for ch in CHANNELS if ch not in cfg["channels"]]
    keep = []
    for i, name in enumerate(SCALAR_NAMES):
        base = name[:-len("__missing")] if name.endswith("__missing") else name
        src, _, feat = base.partition("__")
        if src == "scalar" and not cfg["use_scalars"]:
            continue
        if feat.endswith("_popularity") and not cfg.get("use_popularity", False):
            continue
        if any(feat.startswith(ch + "_") or feat.startswith(f"missing_value_flags_{ch}_") for ch in dropped):
            continue
        keep.append(i)
    return np.array(keep, dtype=np.int64)

30 scalar inputs: 16 from Scalar, 11 Sequence summaries, 3 missing-value flags:
 1. sequence__history_len
 2. sequence__cand_has_metadata
 3. sequence__click_days_since_last
 4. sequence__click_empty
 5. sequence__click_popularity
 6. sequence__cart_days_since_last
 7. sequence__cart_empty
 8. sequence__cart_popularity
 9. sequence__order_days_since_last
10. sequence__order_empty
11. sequence__order_popularity
12. scalar__is_dominant_brand
13. scalar__brand_affinity_ratio
14. scalar__cate_affinity_ratio
15. scalar__hist_item_count
16. scalar__click_is_dominant_brand
17. scalar__click_brand_affinity_ratio
18. scalar__click_cate_affinity_ratio
19. scalar__click_hist_item_count
20. scalar__cart_is_dominant_brand
21. scalar__cart_brand_affinity_ratio
22. scalar__cart_cate_affinity_ratio
23. scalar__cart_hist_item_count
24. scalar__order_is_dominant_brand
25. scalar__order_brand_affinity_ratio
26. scalar__order_cate_affinity_ratio
27. scalar__order_hist_item_count
28. sequence__click_days_s

## Step 3: Batching

A batch is a set of whole searches: each search's histories are encoded once and shared by all of its candidates (`row` points each candidate to its search). Training batches are filled up to `batch_cands` candidates.

In [11]:
class SplitData:
    def __init__(self, split):
        self.split = split
        self.slate_id, self.user, self.seqs, self.cand = RAW[split]
        self.scalars = SCALARS[split]
        n = len(self.slate_id)
        self.counts = np.bincount(self.cand["row"], minlength=n)
        self.starts = np.concatenate([[0], np.cumsum(self.counts)[:-1]])
        self.has_purchase = np.bincount(self.cand["row"], weights=(self.cand["label"] == 3), minlength=n) > 0
        assert np.all(np.diff(self.cand["row"]) >= 0), "candidates must be grouped by search"

    def batches(self, max_cands, shuffle=False, rng=None, slates=None):
        idx = np.arange(len(self.slate_id)) if slates is None else np.asarray(slates)
        if shuffle:
            idx = rng.permutation(idx)
        out, cur, n = [], [], 0
        for s in idx:
            c = self.counts[s]
            if cur and n + c > max_cands:
                out.append(np.array(cur)); cur, n = [], 0
            cur.append(s); n += c
        if cur:
            out.append(np.array(cur))
        return out

    def gather(self, sl, channels, scalar_cols):
        """Numpy arrays for one batch of searches `sl`."""
        lens = self.counts[sl]
        cidx = np.repeat(self.starts[sl], lens) + (np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens))
        return {
            "seq": {ch: {f: self.seqs[ch][f][sl] for f in SEQ_FIELDS} for ch in channels},
            "cand": {f: self.cand[f][cidx] for f in ["item", *ATTR_COLS, "name_terms", "label"]},
            "row": np.repeat(np.arange(len(sl)), lens),
            "scalars": self.scalars[cidx][:, scalar_cols],
            "cidx": cidx,
        }


DATA = {split: SplitData(split) for split in RAW}
for split, d in DATA.items():
    print(f"{split}: {d.has_purchase.sum():,} searches with a purchase "
          f"(of {len(d.slate_id):,}); median {int(np.median(d.counts))} candidates per search")

train: 14,408 searches with a purchase (of 122,083); median 21 candidates per search
val1: 1,945 searches with a purchase (of 17,799); median 90 candidates per search
val2: 1,930 searches with a purchase (of 16,924); median 90 candidates per search
test: 2,023 searches with a purchase (of 17,025); median 90 candidates per search


In [12]:
def to_device(b):
    def t(x, float_=False):
        x = torch.from_numpy(np.ascontiguousarray(x))
        return (x.float() if float_ else x.long()).to(DEVICE, non_blocking=True)
    return {
        "seq": {ch: {f: t(v, float_=(f == "days")) for f, v in s.items()} for ch, s in b["seq"].items()},
        "cand": {f: t(v) for f, v in b["cand"].items()},
        "row": t(b["row"]),
        "scalars": t(b["scalars"], float_=True),
    }

## Step 4: The model

Following Figure 4 of the report:

1. **Shared embeddings.** A product is the sum of its item, brand, L4 category and shop embeddings (item only if `use_attributes=False`). Query and name words share one table; a query or name is the mean of its word embeddings (padding excluded). Each past action is a linear projection of [product, action type, time bucket, query, search flag].
2. **Candidate vector**: a linear projection of [product, name].
3. **Target attention, one block per channel.** For each candidate, each past action in the channel gets a score from a small network over [action, candidate, action − candidate, action × candidate] (DIN). With `use_decay`, the score is shifted by −ln 2 × days / half-life, which multiplies the attention weight by that channel's recency decay. Padding is masked; weights are normalised with a softmax; an empty channel gives a zero summary (its emptiness is a scalar input).
4. **Output.** [channel summaries, candidate vector, scalar features] → MLP → one logit. The training target is "any interaction", so the logit is an engagement score; Step 6 calibrates it to a purchase probability.

In [13]:
class Model2(nn.Module):
    def __init__(self, cfg, n_scalar):
        super().__init__()
        d, h = cfg["emb_dim"], cfg["att_hidden"]
        self.cfg = cfg
        self.item = nn.Embedding(VOCAB["n_items"], d, padding_idx=0)
        self.attr = nn.ModuleDict({c: nn.Embedding(VOCAB[f"n_{c}"], d, padding_idx=0) for c in ATTR_COLS})
        self.term = nn.Embedding(VOCAB["n_terms"], d, padding_idx=0)
        self.action = nn.Embedding(VOCAB["n_actions"], d, padding_idx=0)
        self.tbucket = nn.Embedding(VOCAB["n_time_buckets"], d, padding_idx=0)
        self.emb_drop = nn.Dropout(cfg.get("emb_dropout", 0.0))
        self.event = nn.Linear(4 * d + 1, d)
        self.cand = nn.Linear(2 * d, d)
        self.att = nn.ModuleDict({ch: nn.Sequential(nn.Linear(4 * d, h), nn.ReLU(), nn.Linear(h, 1))
                                  for ch in cfg["channels"]})
        layers, width = [], len(cfg["channels"]) * d + d + n_scalar
        for out in cfg["mlp_hidden"]:
            layers += [nn.Linear(width, out), nn.ReLU(), nn.Dropout(cfg["dropout"])]
            width = out
        self.mlp = nn.Sequential(*layers, nn.Linear(width, 1))
        self.embedding_tables = [self.item, *self.attr.values(), self.term]
        for emb in [self.item, *self.attr.values(), self.term, self.action, self.tbucket]:
            nn.init.normal_(emb.weight, std=0.05)
            with torch.no_grad():
                emb.weight[0].zero_()

    def product(self, x, use_item=True):
        v = self.item(x["item"]) if use_item else 0.0
        if self.cfg["use_attributes"]:
            for c in ATTR_COLS:
                v = v + self.attr[c](x[c])
        if not torch.is_tensor(v):          # neither item ID nor attributes: a zero vector
            v = torch.zeros(*x["item"].shape, self.item.embedding_dim, device=x["item"].device)
        return self.emb_drop(v)

    def words(self, terms):
        e = self.emb_drop(self.term(terms))
        m = (terms > 0).unsqueeze(-1).to(e.dtype)
        return (e * m).sum(-2) / m.sum(-2).clamp(min=1.0)

    def forward(self, b, return_attn=False):
        c = b["cand"]
        name = self.words(c["name_terms"]) if self.cfg["use_name_terms"] else \
               torch.zeros(len(c["item"]), self.item.embedding_dim, device=c["item"].device)
        cvec = self.cand(torch.cat([self.product(c, use_item=self.cfg.get("use_cand_item_id", True)), name], -1))  # [N, d]
        row, parts, attn = b["row"], [], {}
        for ch in self.cfg["channels"]:
            s = b["seq"][ch]
            if self.cfg["use_query_terms"]:
                q = self.words(s["qry_terms"])
            else:
                q = torch.zeros(*s["item"].shape, self.item.embedding_dim, device=s["item"].device)
            ev = self.event(torch.cat([self.product(s), self.action(s["action"]), self.tbucket(s["time_bucket"]),
                                       q, s["has_query"].unsqueeze(-1).float()], -1))  # [B, L, d]
            H = ev[row]                                                               # [N, L, d]
            C = cvec.unsqueeze(1).expand_as(H)
            logits = self.att[ch](torch.cat([H, C, H - C, H * C], -1)).squeeze(-1).float()
            if self.cfg["use_decay"]:
                logits = logits - math.log(2) * s["days"][row] / HALF_LIFE[ch]
            mask = s["action"][row] > 0
            w = torch.softmax(logits.masked_fill(~mask, -1e9), dim=-1) * mask.float()  # empty channel -> all 0
            parts.append((w.unsqueeze(-1) * H.float()).sum(1))
            if return_attn:
                attn[ch] = w.detach()
        x = torch.cat(parts + [cvec.float(), b["scalars"]], -1)
        out = self.mlp(x).squeeze(-1)
        return (out, attn) if return_attn else out


WEIGHT_T = torch.tensor([GRADED_WEIGHT[k] for k in range(4)], device=DEVICE)


def weighted_bce(logit, label):
    """Graded objective: target = any interaction, weight = 1 / 1 / 5 / 10 for labels 0 / 1 / 2 / 3."""
    w = WEIGHT_T[label]
    y = (label >= 1).float()
    loss = (F.binary_cross_entropy_with_logits(logit.float(), y, reduction="none") * w).sum() / w.sum()
    return loss, float(w.sum())

## Step 5: Metrics

`per_slate_metrics` and `paired_diff` are copied unchanged from the Model 1 / LightGBM ablation notebooks, so every model in the report is scored identically: purchase relevance (label 3), per-search NDCG@10, Hit@10, Hit@1 and MRR, ties broken randomly and averaged over 5 tie-breaks, searches without a purchase excluded.

In [14]:
METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True,
                      n_tiebreaks=5, seed=0):
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)
    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))
        g = gain[order]
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first
    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def paired_diff(a, b, col="ndcg@10", n_boot=2000, seed=0):
    # mean of a - b over slates, with a 95% bootstrap interval (a and b: per-slate tables on the same slates)
    assert (a["slate_id"].to_numpy() == b["slate_id"].to_numpy()).all(), "slates not aligned"
    d = a[col].to_numpy() - b[col].to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0,len(d),size=(min(100,n_boot-i),len(d)))].mean(axis=1)
                           for i in range(0,n_boot,100)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"diff": d.mean(), "ci_low": lo, "ci_high": hi}


def candidate_frame(data, scores, slates=None):
    """slate_id / position / label / score for the candidates of `slates` (default: all)."""
    rows = np.arange(len(data.slate_id)) if slates is None else np.asarray(slates)
    lens = data.counts[rows]
    cidx = np.repeat(data.starts[rows], lens) + (np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens))
    return pd.DataFrame({"slate_id": data.slate_id[data.cand["row"][cidx]], "position": data.cand["pos"][cidx],
                         "label": data.cand["label"][cidx], "score": scores[cidx]})

In [15]:
@torch.no_grad()
def predict(model, data, slates=None):
    """Raw logits for every candidate of `slates` (others NaN), in the data's candidate order."""
    model.eval()
    cols = scalar_columns(model.cfg)
    scores = np.full(len(data.cand["item"]), np.nan, dtype=np.float64)
    for sl in data.batches(model.cfg["eval_batch_cands"], slates=slates):
        b = data.gather(sl, model.cfg["channels"], cols)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=model.cfg["amp"] and DEVICE.type == "cuda"):
            out = model(to_device(b))
        scores[b["cidx"]] = out.float().cpu().numpy()
    return scores


def evaluate(model, data, slates=None):
    """Per-search metrics on searches with a purchase."""
    slates = np.flatnonzero(data.has_purchase) if slates is None else slates
    return per_slate_metrics(candidate_frame(data, predict(model, data, slates), slates), "score")

### Quick checks before training

Runs in about a minute and stops with an error if something is wrong:

1. one forward and backward pass on a few searches: right output shape, no NaNs, gradients reach the embeddings;
2. padding has no effect: appending an extra padded position to every channel leaves the output unchanged;
3. the model can learn: on 64 training searches, 150 steps must cut the training loss by at least half.

In [16]:
def make_optimizer(model, cfg):
    """AdamW. Embedding tables (items, brands, categories, shops, words) get `emb_weight_decay`;
    everything else gets `weight_decay`. With both at 0 this is plain Adam."""
    emb_ids = {id(p) for t in model.embedding_tables for p in t.parameters()}
    emb = [p for p in model.parameters() if id(p) in emb_ids]
    rest = [p for p in model.parameters() if id(p) not in emb_ids]
    return torch.optim.AdamW([{"params": emb, "weight_decay": cfg.get("emb_weight_decay", 0.0)},
                              {"params": rest, "weight_decay": cfg["weight_decay"]}], lr=cfg["lr"])


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu():
    """Release GPU memory from models that are no longer referenced."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _check_forward_backward(cfg, cols):
    set_seed(0)
    m = Model2(cfg, len(cols)).to(DEVICE)
    b = DATA["train"].gather(np.arange(8), cfg["channels"], cols)
    bt = to_device(b)
    out, attn = m(bt, return_attn=True)
    assert out.shape == (len(b["cidx"]),) and torch.isfinite(out).all(), "bad output"
    for ch, w in attn.items():
        s = w.sum(-1)
        nonempty = (bt["seq"][ch]["action"][bt["row"]] > 0).any(-1)
        assert torch.allclose(s[nonempty], torch.ones_like(s[nonempty]), atol=1e-4), f"{ch}: weights must sum to 1"
        assert (s[~nonempty] == 0).all(), f"{ch}: empty channel must give zero weights"
    loss, _ = weighted_bce(out, bt["cand"]["label"])
    loss.backward()
    assert m.item.weight.grad is not None and m.item.weight.grad.abs().sum() > 0, "no gradient to item embeddings"

    # padding invariance: add one padded position at the front of every channel
    m.eval()
    with torch.no_grad():
        base = m(bt)
        bp = {**bt, "seq": {}}
        for ch, s in bt["seq"].items():
            bp["seq"][ch] = {f: torch.cat([torch.zeros_like(v[:, :1]), v], 1) for f, v in s.items()}
        padded = m(bp)
    assert torch.allclose(base, padded, atol=1e-4), "padding changed the output"


def _check_can_learn(cfg, cols):
    m = Model2(cfg, len(cols)).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    bt = to_device(DATA["train"].gather(np.arange(64), cfg["channels"], cols))
    first = None
    for step in range(150):
        m.train()
        loss, _ = weighted_bce(m(bt), bt["cand"]["label"])
        opt.zero_grad(); loss.backward(); opt.step()
        first = first if first is not None else loss.item()
    print(f"overfit check: loss {first:.4f} -> {loss.item():.4f}")
    assert loss.item() < 0.5 * first, "the model failed to fit 64 searches: check the inputs"


def _check_variant(cfg, variant):
    vc = {**cfg, **variant}
    cols_v = scalar_columns(vc)
    mv = Model2(vc, len(cols_v)).to(DEVICE)
    optv = make_optimizer(mv, vc)
    bv = to_device(DATA["train"].gather(np.arange(8), vc["channels"], cols_v))
    lv, _ = weighted_bce(mv(bv), bv["cand"]["label"])
    optv.zero_grad(); lv.backward(); optv.step()
    assert torch.isfinite(lv), f"bad loss with {variant}"


def smoke_tests(cfg):
    """Each check runs in its own function, so its model is freed before the next one is built
    (a model with its optimizer takes about 2 GB of GPU memory at emb_dim 64)."""
    cols = scalar_columns(cfg)
    _check_forward_backward(cfg, cols); free_gpu()
    _check_can_learn(cfg, cols); free_gpu()
    for variant in [{"use_cand_item_id": False}, {"use_popularity": True}, {"emb_dropout": 0.2},
                    {"emb_weight_decay": 0.1}, {"use_cand_item_id": False, "use_attributes": False}]:
        _check_variant(cfg, variant); free_gpu()
    if torch.cuda.is_available():
        print(f"GPU memory in use after the checks: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print("All quick checks passed (including the overfitting switches).")


smoke_tests(CFG)

overfit check: loss 0.7063 -> 0.0000
GPU memory in use after the checks: 0.02 GB
All quick checks passed (including the overfitting switches).


### The training function

`train_model` trains one Model 2: AdamW (weight decay on the embedding tables only, set by `emb_weight_decay`) with gradient clipping and mixed precision, and the graded loss. It checks val1 NDCG@10 `evals_per_epoch` times per epoch (twice by default, because the best point comes early), stops when `patience` checks in a row bring no improvement, and keeps the weights from the best check. With `fixed_points` it trains for exactly that many checks instead (used by the ablation). It is used by the tuning (Step 6), the final training (Step 7) and the ablation (Step 10).

In [17]:
def make_optimizer(model, cfg):
    """AdamW. Embedding tables (items, brands, categories, shops, words) get `emb_weight_decay`;
    everything else gets `weight_decay`. With both at 0 this is plain Adam."""
    emb_ids = {id(p) for t in model.embedding_tables for p in t.parameters()}
    emb = [p for p in model.parameters() if id(p) in emb_ids]
    rest = [p for p in model.parameters() if id(p) not in emb_ids]
    return torch.optim.AdamW([{"params": emb, "weight_decay": cfg.get("emb_weight_decay", 0.0)},
                              {"params": rest, "weight_decay": cfg["weight_decay"]}], lr=cfg["lr"])


def train_model(cfg, fixed_points=None, verbose=True):
    """Train one Model 2, checking val1 `evals_per_epoch` times per epoch.
    fixed_points=None: early stopping on val1 NDCG@10 (keeps the best check's weights);
    otherwise train exactly that many check intervals (used by the ablation)."""
    set_seed(cfg["seed"])
    cols = scalar_columns(cfg)
    model = Model2(cfg, len(cols)).to(DEVICE)
    opt = make_optimizer(model, cfg)
    use_amp = cfg["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = np.random.default_rng(cfg["seed"])
    tr, va = DATA["train"], DATA[TUNE_SPLIT]
    k = max(1, int(cfg.get("evals_per_epoch", 1)))
    history, best, best_state, best_point, bad, point = [], -1.0, None, 0, 0, 0
    t0, total, wsum = time.time(), 0.0, 0.0
    for epoch in range(1, cfg["max_epochs"] + 1):
        batches = tr.batches(cfg["batch_cands"], shuffle=True, rng=rng)
        parts = np.array_split(np.arange(len(batches)), k)
        for part in parts:
            model.train()
            for bi in part:
                b = to_device(tr.gather(batches[bi], cfg["channels"], cols))
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    out = model(b)
                loss, w = weighted_bce(out, b["cand"]["label"])
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(opt)
                scaler.update()
                total += loss.item() * w; wsum += w
            point += 1
            val = evaluate(model, va)
            rec = {"check": point, "epoch": round(point / k, 2), "train_loss": total / max(wsum, 1e-12),
                   **{f"{TUNE_SPLIT}_{m}": val[m].mean() for m in METRICS}, "minutes": (time.time() - t0) / 60}
            history.append(rec)
            t0, total, wsum = time.time(), 0.0, 0.0
            if verbose:
                print(f"epoch {rec['epoch']:.2f}: train loss {rec['train_loss']:.4f} | "
                      f"{TUNE_SPLIT} NDCG@10 {rec[TUNE_SPLIT + '_ndcg@10']:.4f} | "
                      f"{TUNE_SPLIT} Hit@10 {rec[TUNE_SPLIT + '_hit@10']:.4f} | {rec['minutes']:.1f} min")
            if fixed_points is not None:
                if point >= fixed_points:
                    return model, pd.DataFrame(history), point
                continue
            if rec[f"{TUNE_SPLIT}_ndcg@10"] > best + 1e-4:
                best, best_point, bad = rec[f"{TUNE_SPLIT}_ndcg@10"], point, 0
                best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    model.load_state_dict(best_state)
                    return model, pd.DataFrame(history), best_point
    if fixed_points is None:
        model.load_state_dict(best_state)
        return model, pd.DataFrame(history), best_point
    return model, pd.DataFrame(history), point


TUNED = {}   # filled by the optional tuning step (Step 6)

## Step 6: Reducing overfitting: tuning on val1

The first runs peaked after one epoch on val1 (train NDCG@10 about 0.70 against 0.31), and learning rate, embedding size and dropout made little difference. The likely cause is the item embedding table: about 2 million product vectors, most fitted to a handful of training searches. Each configuration below changes one thing that targets this, starting from the current settings (embedding size 64):

- **embedding weight decay**: pulls product and word vectors that the data does not support towards zero;
- **no candidate item ID**: scores candidates from brand, category, shop and name only, so nothing item-specific can be memorised on the candidate side;
- **embedding dropout**: randomly hides parts of the embeddings during training;
- **candidate popularity**: the signal Model 1 relies on most (not a regulariser, but a likely gain).

Each is trained with early stopping on val1 (checked twice per epoch, one seed) and the best by val1 NDCG@10 becomes the final configuration in Step 7. With one seed, differences below about 0.005 are within noise: treat them as ties and prefer the simpler setting. Val2 and test are not touched. About 2–5 minutes per configuration.

In [19]:
RUN_TUNING = True
N_TRIALS = 20
STUDY_NAME = "tuning_overfitting_30_inputs"   # new name: earlier trials used different inputs (45 scalars)

if RUN_TUNING:
    base_cfg = CFG.copy()
    metric = f"{TUNE_SPLIT}_ndcg@10"
    hit_metric = f"{TUNE_SPLIT}_hit@10"

    def objective(trial):
        params = {
            "emb_weight_decay": trial.suggest_float("emb_weight_decay", 0.1, 1.0, step=0.1),
            "emb_dropout": trial.suggest_float("emb_dropout", 0.1, 0.5, step=0.1),
        }
        config = {**base_cfg, **params}
        t0 = time.time()
        m = hist = None
        try:
            m, hist, best_point = train_model(config, verbose=False)
            best_row = hist.loc[hist[metric].idxmax()]
            # stored with the trial, so the table can be rebuilt even after a disconnect
            trial.set_user_attr("best_epoch", round(best_point / config["evals_per_epoch"], 2))
            trial.set_user_attr(hit_metric, float(best_row[hit_metric]))
            trial.set_user_attr("minutes", (time.time() - t0) / 60)
            print({"trial": trial.number, **params, metric: round(float(best_row[metric]), 4)})
            return float(best_row[metric])
        finally:
            del m, hist
            free_gpu()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=4222),
        storage=f"sqlite:///{OUT_DIR}/tuning.db",
        study_name=STUDY_NAME,
        load_if_exists=True,          # resumes this study if Colab disconnects mid-way
    )
    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - done, n_jobs=1)

    # table from the study itself (all completed trials, including any from before a disconnect)
    df = study.trials_dataframe()
    df = df[df["state"] == "COMPLETE"]
    tuning = pd.DataFrame({
        "trial": df["number"],
        "emb_weight_decay": df["params_emb_weight_decay"],
        "emb_dropout": df["params_emb_dropout"],
        "best_epoch": df.get("user_attrs_best_epoch"),
        metric: df["value"],
        hit_metric: df.get(f"user_attrs_{hit_metric}"),
        "minutes": df.get("user_attrs_minutes"),
    }).sort_values(metric, ascending=False).reset_index(drop=True)
    display(tuning.round(4))
    tuning.to_csv(OUT_DIR / "tuning_overfitting.csv", index=False)

    TUNED = study.best_params
    CFG.update(TUNED)
    print(f"chosen on {TUNE_SPLIT}: {TUNED}")
    print(f"best {metric}: {study.best_value:.4f}")

[I 2026-11-08 06:53:54,793] A new study created in RDB with name: tuning_overfitting_30_inputs


{'trial': 0, 'emb_weight_decay': 0.2, 'emb_dropout': 0.4, 'val1_ndcg@10': 0.3659}


[I 2026-11-08 06:58:37,265] Trial 0 finished with value: 0.3658929194843767 and parameters: {'emb_weight_decay': 0.2, 'emb_dropout': 0.4}. Best is trial 0 with value: 0.3658929194843767.


{'trial': 1, 'emb_weight_decay': 0.8, 'emb_dropout': 0.30000000000000004, 'val1_ndcg@10': 0.3674}


[I 2026-11-08 07:03:20,070] Trial 1 finished with value: 0.3674213956485629 and parameters: {'emb_weight_decay': 0.8, 'emb_dropout': 0.30000000000000004}. Best is trial 1 with value: 0.3674213956485629.


{'trial': 2, 'emb_weight_decay': 0.6, 'emb_dropout': 0.30000000000000004, 'val1_ndcg@10': 0.3667}


... (33 outputs left out)

{'trial': 19, 'emb_weight_decay': 0.30000000000000004, 'emb_dropout': 0.30000000000000004, 'val1_ndcg@10': 0.3671}


[I 2026-11-08 08:28:05,628] Trial 19 finished with value: 0.3670978662777755 and parameters: {'emb_weight_decay': 0.30000000000000004, 'emb_dropout': 0.30000000000000004}. Best is trial 13 with value: 0.36824870954896444.


,trial,emb_weight_decay,emb_dropout,best_epoch,val1_ndcg@10,val1_hit@10,minutes
0,13,0.3,0.2,1.0,0.3682,0.6187,4.7000
1,11,0.3,0.2,1.0,0.3677,0.6172,4.7011
2,17,0.5,0.2,1.0,0.3676,0.6183,4.6890
3,12,0.4,0.2,1.0,0.3676,0.6172,4.7046
4,1,0.8,0.3,1.0,0.3674,0.6155,4.7068
5,7,0.1,0.2,1.0,0.3672,0.6170,4.7191
6,19,0.3,0.3,1.0,0.3671,0.6150,4.6873
7,18,0.6,0.4,1.0,0.3671,0.6151,4.6907
8,5,0.9,0.5,1.0,0.3668,0.6138,4.7011
9,8,0.9,0.1,1.0,0.3668,0.6166,4.7016


chosen on val1: {'emb_weight_decay': 0.30000000000000004, 'emb_dropout': 0.2}
best val1_ndcg@10: 0.3682


## Step 7: Train with early stopping

Each epoch shuffles the training searches; val1 NDCG@10 is checked twice per epoch on the searches with a purchase. Training stops after `patience` checks without improvement, and the best check's weights are kept and saved. If Step 6 was run, `CFG` already holds the configuration chosen there.

In [20]:
free_gpu()
model, history, BEST_POINT = train_model(CFG)
BEST_EPOCH = BEST_POINT / CFG["evals_per_epoch"]
display(history.round(4))
print(f"best check: {BEST_POINT} (epoch {BEST_EPOCH:.2f}) | settings:",
      {k: CFG[k] for k in ["lr", "emb_dim", "dropout", "emb_dropout", "emb_weight_decay", "use_cand_item_id", "use_popularity"]})
torch.save({"state_dict": model.state_dict(), "cfg": CFG, "scalar_names": SCALAR_NAMES,
            "scalar_columns": scalar_columns(CFG).tolist(), "best_point": BEST_POINT},
           OUT_DIR / "model2_din.pt")
history.to_csv(OUT_DIR / "training_history.csv", index=False)

epoch 0.50: train loss 0.4213 | val1 NDCG@10 0.3607 | val1 Hit@10 0.6000 | 0.9 min
epoch 1.00: train loss 0.4059 | val1 NDCG@10 0.3677 | val1 Hit@10 0.6171 | 0.9 min
epoch 1.50: train loss 0.3315 | val1 NDCG@10 0.3227 | val1 Hit@10 0.5537 | 0.9 min
epoch 2.00: train loss 0.3418 | val1 NDCG@10 0.3242 | val1 Hit@10 0.5494 | 0.9 min
epoch 2.50: train loss 0.2517 | val1 NDCG@10 0.2777 | val1 Hit@10 0.5011 | 0.9 min


,check,epoch,train_loss,val1_ndcg@10,val1_hit@10,val1_hit@1,val1_mrr,minutes
0,1,0.5,0.4213,0.3607,0.6000,0.1794,0.3095,0.9269
1,2,1.0,0.4059,0.3677,0.6171,0.1774,0.3117,0.9318
2,3,1.5,0.3315,0.3227,0.5537,0.1476,0.2736,0.9367
3,4,2.0,0.3418,0.3242,0.5494,0.1465,0.2774,0.9229
4,5,2.5,0.2517,0.2777,0.5011,0.1105,0.2323,0.9286


best check: 2 (epoch 1.00) | settings: {'lr': 0.001, 'emb_dim': 64, 'dropout': 0.2, 'emb_dropout': 0.2, 'emb_weight_decay': 0.30000000000000004, 'use_cand_item_id': True, 'use_popularity': True}


## Step 8: Results, calibration and predictions for the gate

- **Performance** on train (searches with a purchase, sampled candidate lists, so optimistic), val1 (used for early stopping and tuning, so slightly optimistic), val2 and test (full lists). Test is scored here once.
- **Calibration** (Platt scaling) is fitted on all **val2** candidates against the *purchase* label, turning the engagement score into a purchase probability P2 comparable with P1. Val2 was not used for model selection. Calibration is monotonic within a search, so it does not change the ranking.
- **Saved for the gate:** `model2_predictions_<split>.parquet` with `slate_id`, `position`, `label`, raw logit and calibrated `p2`, for every val2 candidate (to fit the gate) and test candidate (to evaluate it); per-search metrics for the bootstrap.

In [21]:
scores = {split: predict(model, DATA[split]) for split in ["val1", "val2", "test"]}
scores["train"] = predict(model, DATA["train"], np.flatnonzero(DATA["train"].has_purchase))

rows, per_slate = [], {}
for split in SPLITS:
    d = DATA[split]
    ps = per_slate_metrics(candidate_frame(d, scores[split], np.flatnonzero(d.has_purchase)), "score")
    per_slate[split] = ps
    rows.append({"split": split, "searches": len(ps), **ps[METRICS].mean().to_dict()})
results = pd.DataFrame(rows).set_index("split")
display(results.round(4))
results.to_csv(OUT_DIR / "results.csv")
for split in ["val1", "val2", "test"]:
    per_slate[split].to_parquet(OUT_DIR / f"per_slate_metrics_{split}.parquet", index=False)


def fit_platt(logit, label):
    lr = LogisticRegression(C=1e6, max_iter=1000)
    lr.fit(logit.reshape(-1, 1), (label == 3).astype(int))
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])


def apply_platt(logit, a, b):
    return 1.0 / (1.0 + np.exp(-np.clip(a * logit + b, -50, 50)))


A, B = fit_platt(scores[CALIB_SPLIT], DATA[CALIB_SPLIT].cand["label"])
(OUT_DIR / "calibration.json").write_text(json.dumps({"method": "platt", "a": A, "b": B, "fitted_on": CALIB_SPLIT}, indent=2))
for split in [CALIB_SPLIT, "test"]:
    d = DATA[split]
    p2 = apply_platt(scores[split], A, B)
    pd.DataFrame({"slate_id": d.slate_id[d.cand["row"]], "position": d.cand["pos"], "label": d.cand["label"],
                  "p2_logit": scores[split], "p2": p2}).to_parquet(OUT_DIR / f"model2_predictions_{split}.parquet", index=False)
    is_buy = d.cand["label"] == 3
    print(f"{split}: mean P2 {p2.mean():.5f} vs purchase rate {is_buy.mean():.5f} "
          f"(calibrated on {CALIB_SPLIT}: these should be close there)")

,searches,ndcg@10,hit@10,hit@1,mrr
split,,,,,
train,14408,0.6710,0.9428,0.4152,0.5905
val1,1945,0.3677,0.6171,0.1774,0.3117
val2,1930,0.3626,0.6060,0.1708,0.3073
test,2023,0.3673,0.6139,0.1769,0.3113


val2: mean P2 0.00133 vs purchase rate 0.00131 (calibrated on val2: these should be close there)
test: mean P2 0.00132 vs purchase rate 0.00138 (calibrated on val2: these should be close there)


### Train-set predictions for the evaluation (for step5_gate.ipynb)

Step 8 saves predictions for val2 and test only. Evaluation step of `step5_gate.ipynb` also compares the models on the **train set**, so this cell scores the train set with the trained model and saves the result.

- **Model:** reloaded from `model2_din.pt` (the best check from Step 7, with the configuration chosen in Step 6). Nothing is retrained. The reloaded model has the same weights as the one in memory.
- **`p2`:** the Platt formula fitted on val2 in Step 8 (`calibration.json`), applied unchanged. This cell therefore has to run after the Step 8 cell.
- **Searches:** only searches with at least one purchase are scored and saved, the same 14,408 train searches as the `train` row of `results.csv`. The step 6 metrics use only these searches.
- **Read with care:** train lists are the negative-sampled ones (all positives plus up to 20 negatives, about 21 candidates per search) and the model was fitted on them. Train metrics are therefore optimistic and cannot be compared directly with val2 or test.

**Saved:** `model2_predictions_train.parquet` with `slate_id`, `position`, `label`, `p2_logit`, `p2`, in `models/model2/`.

**Check:** the metrics printed at the end must equal the `train` row of `results.csv` (NDCG@10 0.6814, HitRate@10 0.9441, HitRate@1 0.4317, MRR 0.6035). If they differ, do not use the file.

In [22]:
# train predictions (only searches with a purchase), from the saved model
ck = torch.load(OUT_DIR / "model2_din.pt", map_location=DEVICE, weights_only=False)
cfg_saved = ck["cfg"]
model = Model2(cfg_saved, len(scalar_columns(cfg_saved))).to(DEVICE)
model.load_state_dict(ck["state_dict"]); model.eval()

cal2 = json.loads((OUT_DIR / "calibration.json").read_text())
d = DATA["train"]
logit = predict(model, d, np.flatnonzero(d.has_purchase))              # NaN for searches without a purchase
train_pred = pd.DataFrame({"slate_id": d.slate_id[d.cand["row"]], "position": d.cand["pos"], "label": d.cand["label"],
                           "p2_logit": logit,
                           "p2": 1.0 / (1.0 + np.exp(-np.clip(cal2["a"] * logit + cal2["b"], -50, 50)))})
train_pred = train_pred[train_pred["p2_logit"].notna()].reset_index(drop=True)
train_pred.to_parquet(OUT_DIR / "model2_predictions_train.parquet", index=False)
print("train:", train_pred["slate_id"].nunique(), "searches,", len(train_pred), "rows")

# check: must reproduce the train row of results.csv (NDCG@10 0.6814, HitRate@10 0.9441, HitRate@1 0.4317, MRR 0.6035)
print(per_slate_metrics(train_pred.rename(columns={"p2_logit": "score"}), "score")[METRICS].mean().round(4))

train: 14408 searches, 312235 rows
ndcg@10    0.6710
hit@10     0.9428
hit@1      0.4152
mrr        0.5905
dtype: float64


## Step 9: Diagnostics (val2)

Run on val2, which was not used to choose the model.

**By segment.** NDCG@10 for groups of searches: short histories (at or below the train 10th percentile, the gate's Model-1-only rule), no clicks in the last 7 days, no orders in the last year, and whether the purchased product has a learned item embedding.

**Attention.** For each channel, how much attention a candidate puts on past actions of its *own brand*, compared with the share of such actions in the channel (what uniform attention would give). A ratio above 1 means attention concentrates on relevant history; it is reported separately for purchased and other candidates.

In [23]:
def segment_table(data, ps, train_hist_len):
    sid_to_row = pd.Series(np.arange(len(data.slate_id)), index=data.slate_id)
    r = sid_to_row.loc[ps["slate_id"]].to_numpy()
    hl = data.user["history_len"][r]
    p10, p50 = np.percentile(train_hist_len, [10, 50])
    buy = data.cand["label"] == 3
    known_buy = np.bincount(data.cand["row"], weights=buy & (data.cand["item"] > 1), minlength=len(data.slate_id)) > 0
    seg = {
        f"history ≤ {p10:.0f} actions (gate: Model 1 only)": hl <= p10,
        f"history {p10:.0f}–{p50:.0f}": (hl > p10) & (hl <= p50),
        f"history > {p50:.0f}": hl > p50,
        "no clicks in last 7 days": data.seqs["click"]["full_len"][r] == 0,
        "clicked in last 7 days": data.seqs["click"]["full_len"][r] > 0,
        "no orders in last year": data.seqs["order"]["full_len"][r] == 0,
        "purchased product has a learned embedding": known_buy[r],
        "purchased product unknown (cold)": ~known_buy[r],
    }
    return pd.DataFrame([{"segment": k, "searches": int(m.sum()),
                          **{c: ps.loc[m, c].mean() for c in ["ndcg@10", "hit@10"]}} for k, m in seg.items()]).set_index("segment")


@torch.no_grad()
def attention_brand_focus(model, data, max_searches=500):
    model.eval()
    cols = scalar_columns(model.cfg)
    slates = np.flatnonzero(data.has_purchase)[:max_searches]
    acc = []
    for sl in data.batches(model.cfg["eval_batch_cands"], slates=slates):
        b = data.gather(sl, model.cfg["channels"], cols)
        _, attn = model(to_device(b), return_attn=True)
        cb = b["cand"]["brand_id"]
        for ch, w in attn.items():
            w = w.cpu().numpy()
            hb = b["seq"][ch]["brand_id"][b["row"]]
            real = b["seq"][ch]["action"][b["row"]] > 0
            same = real & (hb == cb[:, None]) & (cb[:, None] > 1)
            ok = same.any(1)                     # candidates with at least one own-brand action in the channel
            acc.append(pd.DataFrame({"channel": ch, "purchased": b["cand"]["label"][ok] == 3,
                                     "attention_on_sequence_brand": (w * same).sum(1)[ok],
                                     "share_of_sequence_brand_actions": same.sum(1)[ok] / real.sum(1)[ok]}))
    df = pd.concat(acc)
    out = df.groupby(["channel", "purchased"]).agg(candidates=("purchased", "size"),
                                                   attention_on_sequence_brand=("attention_on_sequence_brand", "mean"),
                                                   share_of_sequence_brand_actions=("share_of_sequence_brand_actions", "mean"))
    out["ratio"] = out["attention_on_sequence_brand"] / out["share_of_sequence_brand_actions"]
    return out


seg = segment_table(DATA["val2"], per_slate["val2"], DATA["train"].user["history_len"])
display(seg.round(4)); seg.to_csv(OUT_DIR / "segments_val2.csv")
att = attention_brand_focus(model, DATA["val2"])
display(att.round(4)); att.to_csv(OUT_DIR / "attention_brand_focus_val2.csv")

,searches,ndcg@10,hit@10
segment,,,
history ≤ 10 actions (gate: Model 1 only),158,0.3153,0.5000
history 10–78,692,0.3595,0.6017
history > 78,1080,0.3715,0.6243
no clicks in last 7 days,341,0.3613,0.5924
clicked in last 7 days,1589,0.3629,0.6089
no orders in last year,57,0.3075,0.5088
purchased product has a learned embedding,1556,0.4343,0.7192
purchased product unknown (cold),374,0.0645,0.1353


candidates  attention_on_sequence_brand  \
channel purchased                                            
cart    False            4544                       0.1810   
        True              100                       0.1768   
click   False            4109                       0.3160   
        True               88                       0.3193   
order   False            3504                       0.1982   
        True               88                       0.1883   

                   share_of_sequence_brand_actions   ratio  
channel purchased                                           
cart    False                               0.1650  1.0969  
        True                                0.1567  1.1282  
click   False                               0.2900  1.0895  
        True                                0.2881  1.1080  
order   False                               0.1868  1.0612  
        True                                0.1783  1.0560

## Hand-off to the gate

- **P2:** `model2_predictions_val2.parquet` (to fit the gate, in `models/model2/`) and `model2_predictions_test.parquet` (final evaluation): `slate_id`, `position`, `label`, `p2_logit`, calibrated `p2`. Join with Model 1's predictions on `(slate_id, position)`.
- **Per-search metrics:** `per_slate_metrics_<split>.parquet`, for paired bootstraps against Model 1 and the blend.
- **Report tables:** `results.csv` (train / val1 / val2 / test), `segments_val2.csv`, `attention_brand_focus_val2.csv`, `training_history.csv`, `tuning_overfitting.csv` if Step 6 was run, and `ablation_din.csv` if Step 10 was run.